<a href="https://colab.research.google.com/github/eduardozurek/ia/blob/main/_007_aprendizaje_supervisado/_003_Aprendizaje_de_Forma_Normal_Disyuntiva_v_SEPT_17_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Generación de una Hipótesis de Forma Normal Disyuntiva a partir de Datos Booleanos**
# Eduardo Zurek, Ph.D.
# Inteligencia Artificial
# Programa de Ingeniería de Sistemas
# Universidad del Norte


## Forma Normal Disjuntiva
$ (A \wedge B) \vee (C \wedge D)$

## Conjunto de Datos Booleanos
Sea un conjunto de datos Booleanos:\
$C={(X_0,y_0),(X_1,y_1),...,(X_n,y_n)}$\
Donde:\
$X_i=<f_{0i},f_{1i},...,f_{mi}>$

Por ejemplo:

|Tupla|$f_0$|$f_1$|$f_2$|$f_3$|$y$|
|-|-|-|-|-|-|
|$X_0$|0|0|0|0|0|
|$X_1$|0|0|1|0|0|
|$X_2$|1|0|0|1|1|
|$X_3$|0|1|1|0|1|
|$X_4$|0|1|0|0|0|
|$X_5$|0|0|1|1|0|
|$X_6$|1|1|0|0|0|
|$X_7$|0|1|1|1|1|
|$X_8$|1|1|0|1|1|

La idea es generar una hipótesis\
$h(X)$ para clasificar $X$ en las\
categorías de $y$


Vale mencionar que otro algoritmo importante para obtener la Forma Normal Disyuntiva es el de [Quine-MacCluskey](https://archive.org/details/bstj35-6-1417/page/n23/mode/2up).

## **El Algoritmo**

$P$ es el conjunto de la tuplas para las cuales $y=1$

$h\leftarrow0$ (falso) es el valor inicial de la hipótesis

**Haga hasta** que $P$ quede vacío
>$r\leftarrow1$ (verdadero)\
>$N$ es el conjunto de las tuplas que no cumplen $r$\
>**Haga hasta** que $N$ quede vacío
>>**Si** todas las características ya están en $r$:
>>>Falla, no hay manera de remover elementos de $N$

>>**De otro modo (sino)**
>>>Seleccione una caracterísca $x_j$ y adicionela a $r$: $r \leftarrow r \wedge x_j$\
Se qitan de $N$ las tuplas en las cuales $x_j=0$

>$h \leftarrow h \vee r$\
$Cub$ es el conjunto formado por las tuplas en $P$ que son cubiertas por $r$\
**Si** $Cub$ está vacío: Falla, no hay manera de actualizar $P$\
**De otro modo**
>>Se retiran de $P$ los elementos de $Cub$


## Heurística de Selección
La heurística que se aplica para seleccionar la $x_j$ con que se actualiza $r$, es la siguiente:

$v_j=n_j^+/max(n_j^-,0.001)$

Donde:

$n_j^+$ es el número de elementos $P$ que SÍ son cubiertos por la hipótesis $r \wedge x_j$

$n_j^-$ es el número de elementos de $N$ que NO son cubiertos por la hipótesis $r \wedge x_j$



### Referencias
Este algoritmo ha sido tomado de MIT-OCW:\
6.034-Artificial Intelligence (Spring 2005)-Undergraduate\
Chapter 4: Learning Introduction\
https://ocw.mit.edu/courses/electrical-engineering-and-computer-science/6-034-artificial-intelligence-spring-2005/lecture-notes/ch4_learnintro.pdf

## **Explicación con ejemplo paso a paso**

Sea la siguiente tabla de verdad:

|Tupla|$f_0$|$f_1$|$f_2$|$f_3$|$y$|
|-|-|-|-|-|-|
|$X_0$|0|0|0|0|0|
|$X_1$|0|0|1|0|0|
|$X_2$|1|0|0|1|1|
|$X_3$|0|1|1|0|1|
|$X_4$|0|1|0|0|0|
|$X_5$|0|0|1|1|0|
|$X_6$|1|1|0|0|0|
|$X_7$|0|1|1|1|1|
|$X_8$|1|1|0|1|1|

$P$ es el conjunto de la tuplas para las cuales $y=1$

$h\leftarrow0$ (falso) es el valor inicial de la hipótesis


$v_j=n_j^+/max(n_j^-,0.001)$

Donde:

$n_j^+$ es el número de elementos $P$ que SÍ son cubiertos por la hipótesis $r \wedge x_j$

$n_j^-$ es el número de elementos de $N$ que NO son cubiertos por la hipótesis $r \wedge x_j$

## **La Implementación**
El script que les presento a continuación contiene una implementación del algoritmo descrito en las celdas anteriores.


In [1]:
try:
    import numpy as np
except ImportError:
    %pip install numpy
    import numpy as np

import copy

from numpy import logical_and as npand
from IPython.display import display, Math


# ---------------------------------------------------------
# Funciones auxiliares
# ---------------------------------------------------------

def vector(v):
    """Convierte un vector booleano/binario a texto."""
    v = np.asarray(v, dtype=int)
    return "[" + " ".join(map(str, v)) + "]"


def mostrar_vector(nombre, v):
    display(Math(rf"{nombre} = {vector(v)}"))


# ---------------------------------------------------------
# Funciones del algoritmo
# ---------------------------------------------------------

def n_mas(xp, yp, r, j):

    rf = npand(r, xp)

    nmas = np.sum(
        npand(rf, yp)
    )

    display(Math(
        rf"r \land f_{{{j}}} = {vector(rf)}"
    ))

    display(Math(
        rf"y = {vector(yp)}"
    ))

    display(Math(
        rf"n_{{{j}}}^+ = {nmas}"
    ))

    return nmas


def n_menos(xn, yn, r, j):

    rf = npand(r, xn)

    nmenos = np.max([
        np.sum(
            np.logical_xor(rf, yn)
        ),
        0.001
    ])

    display(Math(
        rf"r \land f_{{{j}}} = {vector(rf)}"
    ))

    display(Math(
        rf"y = {vector(yn)}"
    ))

    display(Math(
        rf"n_{{{j}}}^- = {nmenos}"
    ))

    return nmenos


def v(X, y, ip, indn, r, j):

    print()
    print("─" * 60)
    print(f"Evaluando f{j}")
    print("─" * 60)

    print("\nConjunto positivo:")

    nplus = n_mas(
        X[j, ip],
        y[ip],
        r[ip],
        j
    )

    print("\nConjunto negativo:")

    nminus = n_menos(
        X[j, indn],
        y[indn],
        r[indn],
        j
    )

    vj = nplus / nminus

    display(Math(
        rf"v_{{{j}}}"
        rf"="
        rf"\frac{{n_{{{j}}}^+}}{{n_{{{j}}}^-}}"
        rf"="
        rf"\frac{{{nplus}}}{{{nminus}}}"
        rf"="
        rf"{vj:.3f}"
    ))

    return vj


# ---------------------------------------------------------
# Datos
# ---------------------------------------------------------

X = np.array([
    [0,0,1,0,0,0,1,0,1],
    [0,0,0,1,1,0,1,1,1],
    [0,1,0,1,0,1,0,1,0],
    [0,0,1,0,0,1,0,1,1]
])

y = np.array(
    [0,0,1,1,0,0,0,1,1]
)


# ---------------------------------------------------------
# Algoritmo
# ---------------------------------------------------------

print("Dimensiones de la matriz")

display(Math(
    rf"X \in \mathbb{{R}}^{{{X.shape[0]}\times {X.shape[1]}}}"
))


ip = y == 1

print("\nConjunto positivo inicial")

mostrar_vector(
    "P",
    ip
)


h = np.zeros(len(y))
h_lista = []


while np.sum(ip) > 0:

    r = np.ones(
        len(y),
        dtype=bool
    )

    r_lista = []

    indn = np.logical_xor(
        r,
        y
    )

    print("\n")
    print("═" * 60)
    print("CONSTRUCCIÓN DE UNA NUEVA REGLA")
    print("═" * 60)

    mostrar_vector(
        "P",
        ip
    )

    mostrar_vector(
        "N",
        indn
    )


    while np.sum(indn) > 0:

        if len(r_lista) == X.shape[0]:

            print("\nNo tiene solución")
            break

        else:

            v_max = 0
            j_max = 0

            for j in range(X.shape[0]):

                v_j = v(
                    X,
                    y,
                    ip,
                    indn,
                    r,
                    j
                )

                if v_j > v_max:

                    v_max = copy.deepcopy(
                        v_j
                    )

                    j_max = copy.deepcopy(
                        j
                    )


            print()
            print("═" * 60)
            print("Variable seleccionada")
            print("═" * 60)

            display(Math(
                rf"j^* = {j_max}"
            ))

            display(Math(
                rf"v_{{j^*}} = {v_max:.3f}"
            ))


            r_lista.append(
                j_max
            )


            r = npand(
                r,
                X[j_max]
            )


            indn = npand(
                indn,
                X[j_max] == 1
            )


            print("\nActualización de la regla")

            display(Math(
                rf"r \leftarrow r \land f_{{{j_max}}}"
            ))

            mostrar_vector(
                "r",
                r
            )


            print("\nNegativos todavía cubiertos")

            mostrar_vector(
                "N",
                indn
            )


    print()
    print("═" * 60)
    print("Regla obtenida")
    print("═" * 60)


    mostrar_vector(
        "r",
        r
    )


    if len(r_lista) > 0:

        regla = r" \land ".join(
            [
                rf"f_{{{j}}}"
                for j in r_lista
            ]
        )

        display(Math(
            rf"r = {regla}"
        ))


    h_lista.append(
        r_lista
    )


    Cub = npand(
        ip,
        r
    )


    print("\nPositivos cubiertos por la regla")

    display(Math(
        r"Cov = P \land r"
    ))

    mostrar_vector(
        "Cov",
        Cub
    )


    if np.sum(Cub) == 0:

        print("\nNo tiene solución")
        break

    else:

        ip = npand(
            ip,
            np.logical_not(Cub)
        )


        print("\nActualización del conjunto positivo")

        display(Math(
            r"P \leftarrow P \land \neg Cov"
        ))

        mostrar_vector(
            "P",
            ip
        )


    print()
    print("═" * 60)
    print("Hipótesis actual")
    print("═" * 60)


    reglas = []

    for regla_indices in h_lista:

        if len(regla_indices) > 0:

            regla = r" \land ".join(
                [
                    rf"f_{{{j}}}"
                    for j in regla_indices
                ]
            )

            reglas.append(
                rf"\left({regla}\right)"
            )


    if len(reglas) > 0:

        hipotesis = r" \lor ".join(
            reglas
        )

        display(Math(
            rf"h = {hipotesis}"
        ))

Dimensiones de la matriz


<IPython.core.display.Math object>


Conjunto positivo inicial


<IPython.core.display.Math object>



════════════════════════════════════════════════════════════
CONSTRUCCIÓN DE UNA NUEVA REGLA
════════════════════════════════════════════════════════════


...


Actualización del conjunto positivo


<IPython.core.display.Math object>

<IPython.core.display.Math object>


════════════════════════════════════════════════════════════
Hipótesis actual
════════════════════════════════════════════════════════════


<IPython.core.display.Math object>